<a href="https://colab.research.google.com/github/AktanM11/AI-OI/blob/main/DAY3_WEEK2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install qdrant-client


In [ ]:
!pip install sentence-transformers

In [7]:
import json
import uuid
import os
from typing import List, Dict, Generator
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, VectorParams, Distance
from sentence_transformers import SentenceTransformer

In [10]:
FILE_PATH = "policies.jsonl"
COLLECTION_NAME = "policies_collection"
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
BATCH_SIZE = 32
encoder = SentenceTransformer("intfloat/multilingual-e5-large")
VECTOR_DIMENSION = encoder.get_sentence_embedding_dimension()

qdrant_client = QdrantClient(url=ENDPOINT, api_key=API_KEY)

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

/tmp/ipykernel_2996/4245804864.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  VECTOR_DIMENSION = encoder.get_sentence_embedding_dimension()


In [11]:
def recursive_chunking(text: str, chunk_size: int = 500, overlap: int = 50) -> List[str]:
    """
    Разбивает текст на куски заданного размера с контролируемым перекрытием.
    Рекурсивно подстраивается, если кусок текста не требует деления.
    """
    if len(text) <= chunk_size:
        return [text] if text.strip() else []


    chunk = text[:chunk_size]

    slice_zone = chunk[-overlap:]
    last_space = slice_zone.rfind(" ")

    if last_space != -1:
        cut_index = (chunk_size - overlap) + last_space + 1
    else:
        cut_index = chunk_size

    final_chunk = text[:cut_index].strip()

    remaining_text = text[cut_index - overlap:]

    return [final_chunk] + recursive_chunking(remaining_text, chunk_size, overlap)

In [12]:
def read_policies(file_path: str) -> Generator[str, None, None]:
    """Построчно читает jsonl файл и возвращает сырой текст политики."""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Файл {file_path} не найден")

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data = json.loads(line)
                if "content" in data:
                    yield data["content"]

In [13]:
def main():
    if not qdrant_client.collection_exists(COLLECTION_NAME):
        print(f"Создание коллекции '{COLLECTION_NAME}'...")
        qdrant_client.create_collection(
            collection_name=COLLECTION_NAME,
            vectors_config=VectorParams(size=VECTOR_DIMENSION, distance=Distance.COSINE),
        )
    else:
        print(f"Коллекция '{COLLECTION_NAME}' уже существует.")

    points_batch = []
    total_chunks = 0

    for raw_text in read_policies(FILE_PATH):

        chunks = recursive_chunking(raw_text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)

        for chunk in chunks:
            if not chunk.strip():
                continue

            text_for_embedding = f"passage: {chunk}"
            vector = encoder.encode(text_for_embedding).tolist()

            point = PointStruct(
                id=str(uuid.uuid4()),
                vector=vector,
                payload={"page_content": chunk}
            )

            points_batch.append(point)
            total_chunks += 1

            if len(points_batch) >= BATCH_SIZE:
                qdrant_client.upsert(
                    collection_name=COLLECTION_NAME,
                    points=points_batch
                )
                print(f"Загружен батч из {len(points_batch)} точек. Всего обработано: {total_chunks}")
                points_batch = []

    if points_batch:
        qdrant_client.upsert(
            collection_name=COLLECTION_NAME,
            points=points_batch
        )
        print(f"Загружен финальный батч из {len(points_batch)} точек. Всего обработано: {total_chunks}")

    print(f"Конвейер успешно завершен. Всего точек в базе: {total_chunks}")


if __name__ == "__main__":
    main()

Создание коллекции 'policies_collection'...
Загружен батч из 32 точек. Всего обработано: 32
Загружен батч из 32 точек. Всего обработано: 64
Загружен батч из 32 точек. Всего обработано: 96
Загружен батч из 32 точек. Всего обработано: 128
Загружен батч из 32 точек. Всего обработано: 160
Загружен финальный батч из 17 точек. Всего обработано: 177
Конвейер успешно завершен. Всего точек в базе: 177


In [14]:
if __name__ == "__main__":
    main()

Коллекция 'policies_collection' уже существует.
Загружен батч из 32 точек. Всего обработано: 32
Загружен батч из 32 точек. Всего обработано: 64
Загружен батч из 32 точек. Всего обработано: 96
Загружен батч из 32 точек. Всего обработано: 128
Загружен батч из 32 точек. Всего обработано: 160
Загружен финальный батч из 17 точек. Всего обработано: 177
Конвейер успешно завершен. Всего точек в базе: 177
